# 06 — Strategy-Score Ablation

`05_rhetorical_scoring.ipynb` scored every comment on 8 rhetorical strategies and joined the scores onto the working dataset (`results/05_cmv_comments_df.csv`). This notebook asks whether adding those scores as features actually improves the model, and whether the effect is real once comment length is controlled for.

Three questions, in order:
1. **Ablation** — added on top of a fixed architecture, how much do strategy scores move PR-AUC relative to the model already selected in `04.1`/`04.2`, and how does that compare to what each individual engineered-feature family adds on top of the strategy scores alone?
2. **Model selection** — with strategy scores available as a candidate feature, does a different (classifier × resampling × feature-set) combination now win, or does the architecture already selected in `04.1`/`04.2` still hold?
3. **Length control** — does any of this survive once the length confound (`04.3_length_controlled_analysis.ipynb`) is controlled for, or is it riding on the same length signal that inflated earlier engineered-feature and TF-IDF results?

## Setup

In [1]:
import pandas as pd

comments_df = pd.read_csv('../results/05_cmv_comments_df.csv.zip')
comments_df = comments_df.copy()  # consolidate blocks - avoids a pandas PerformanceWarning on later single-column inserts

STRATEGY_NAMES = ['justification', 'concession', 'evidence', 'refutation',
                   'personal_narrative', 'call_to_action', 'simplification', 'question_asking']

print(f'Loaded {len(comments_df)} rows, {comments_df.shape[1]} columns')

Loaded 2753 rows, 246 columns


In [2]:
import hashlib

comments_df['thread_id'] = comments_df['original_post'].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)

print(f'Unique threads : {comments_df["thread_id"].nunique()}')
print(f'Total comments : {len(comments_df)}')

Unique threads : 400
Total comments : 2753


In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder

# Same construction as 04.2/04.3, with word_count now included in numeric_features (formalized
# as a feature - see 04.3's closing section) since it's already a column in 05's output.
numeric_features = [
    'sentiment_llm', 'adj_adv_ratio', 'fk_grade_level', 'fk_reading_ease', 'evidence_count', 'word_count',
    'ethos_llm', 'pathos_llm', 'logos_llm',
    'sentiment_llm_op', 'adj_adv_ratio_op', 'fk_grade_level_op', 'evidence_count_op',
    'ethos_llm_op', 'pathos_llm_op', 'logos_llm_op',
    'sentiment_diff', 'ethos_diff', 'pathos_diff', 'logos_diff',
    'adj_adv_ratio_diff', 'fk_grade_level_diff', 'evidence_count_diff',
]
embedding_features = [c for c in comments_df.columns if c.startswith('embedding_')]
label = 'is_convincing'

categorical_features = ['tone_google', 'tone_google_op']
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
tone_encoded = onehot_encoder.fit_transform(comments_df[categorical_features].fillna('unknown'))
tone_df = pd.DataFrame(tone_encoded, columns=onehot_encoder.get_feature_names_out(), index=comments_df.index)

comments_df['is_formal']    = (comments_df['style_google']    == 'Formal').astype(int)
comments_df['is_formal_op'] = (comments_df['style_google_op'] == 'Formal').astype(int)

match_features = ['sentiment_match', 'tone_match', 'style_match', 'ethos_match', 'pathos_match', 'logos_match']
match_df = comments_df[match_features].astype(int)

X = pd.concat([tone_df,
               comments_df[['is_formal', 'is_formal_op', 'use_of_persuasive_lang'] + numeric_features],
               match_df,
               comments_df[embedding_features]],
              axis=1)
X_strategy = comments_df[STRATEGY_NAMES]
y = comments_df[label]
groups = comments_df['thread_id']

print(f'Engineered features : {X.shape[1]}  ({len(embedding_features)} of them PCA embedding components)')
print(f'Strategy features    : {X_strategy.shape[1]}')
print(f'Samples              : {X.shape[0]}')
print(f'Positive rate        : {y.mean():.3f}')

Engineered features : 229  (187 of them PCA embedding components)
Strategy features    : 8
Samples              : 2753
Positive rate        : 0.233


In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_threads = set(groups.iloc[train_idx])
test_threads  = set(groups.iloc[test_idx])
assert len(train_threads & test_threads) == 0, 'LEAKAGE: shared threads between train and test!'
print(f'Train: {len(train_idx)} comments from {len(train_threads)} threads')
print(f'Test : {len(test_idx)} comments from {len(test_threads)} threads')

Train: 2161 comments from 320 threads
Test : 592 comments from 80 threads


The fixed architecture used throughout this notebook — `XGB (over) | trusted+untrusted` — is the same one selected in `04.1_model_training.ipynb` and reconfirmed with `word_count` included in `04.2_baseline_comparison.ipynb`'s closing section. Refit here (self-contained, same convention as `04.3`) rather than depending on those notebooks having been run first.

In [5]:
from sklearn.preprocessing import RobustScaler
import xgboost as xgb
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

def make_selected_pipe():
    return Pipeline([
        ('scaler', RobustScaler()),
        ('sampler', SMOTE(random_state=42)),
        ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss')),
    ])

def bootstrap_ci(y_true, proba, metric_fn, n_boot=1000, seed=42):
    """95% CI for a metric_fn(y_true_sample, proba_sample) via resampling with replacement."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    proba  = np.asarray(proba)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, pb = y_true[idx], proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        scores.append(metric_fn(yt, pb))
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

## Ablation

This section isolates strategy scores' marginal contribution with the simplest comparison that answers it directly: the model already selected in `04.1`/`04.2` (`X`, 229 columns — `trusted+untrusted`), that same model with the 8 strategy scores added (`X_all`, 237 columns), and the strategy scores alone, isolating their standalone signal. The architecture is held fixed (`XGB (over)`, the pipeline selected in `04.1`/`04.2`) throughout — only the feature set changes.

In [6]:
STRATEGY_COLS = STRATEGY_NAMES
assert set(STRATEGY_COLS) <= set(X_strategy.columns)

X_all = pd.concat([X, X_strategy], axis=1)

print(f'Current model (X)          : {X.shape[1]:3d} columns')
print(f'Current model + strategies : {X_all.shape[1]:3d} columns')
print(f'Strategy scores only       : {len(STRATEGY_COLS):3d} columns')

Current model (X)          : 229 columns
Current model + strategies : 237 columns
Strategy scores only       :   8 columns


In [7]:
from sklearn.base import clone

def evaluate_ablation_stage(cols):
    """Fit the fixed selected-pipeline architecture on the given columns only, train/test split
    already carved above. Returns threshold-free metrics plus bootstrap 95% CIs."""
    X_train_stage = X_all[cols].iloc[train_idx]
    X_test_stage  = X_all[cols].iloc[test_idx]

    pipe = clone(make_selected_pipe())
    pipe.fit(X_train_stage, y_train)
    proba = pipe.predict_proba(X_test_stage)[:, 1]

    pr_auc  = average_precision_score(y_test, proba)
    roc_auc = roc_auc_score(y_test, proba)
    pr_ci   = bootstrap_ci(y_test, proba, average_precision_score)

    return {'PR-AUC': round(pr_auc, 4), 'PR-AUC 95% CI': pr_ci, 'ROC-AUC': round(roc_auc, 4)}

### A Column-Order Sensitivity Check

Every comparison in this notebook depends on PR-AUC numbers being stable under this specific fixed architecture (`XGB (over)` + `SMOTE`). Before trusting a single-order PR-AUC for any of them, it's worth checking directly whether they're even stable: refit the exact same 237 columns (`X_all` — everything in `X` plus the 8 strategy scores), changing nothing but the order the columns are passed in.

In [8]:
import random

# Same 237 columns (X_all) every time - only their order changes. 5 random shuffles, plus the
# original concat order, so a single lucky/unlucky pair of orderings can't misrepresent the effect.
order_pr_aucs = [evaluate_ablation_stage(list(X_all.columns))['PR-AUC']]
for seed in range(5):
    shuffled = list(X_all.columns)
    random.Random(seed).shuffle(shuffled)
    order_pr_aucs.append(evaluate_ablation_stage(shuffled)['PR-AUC'])

print('PR-AUC across 6 orderings of the identical 237 columns:', order_pr_aucs)
print(f'Range: {max(order_pr_aucs) - min(order_pr_aucs):.4f}  (min {min(order_pr_aucs)}, max {max(order_pr_aucs)})')

PR-AUC across 6 orderings of the identical 237 columns: [0.4968, 0.539, 0.5057, 0.524, 0.5251, 0.5167]
Range: 0.0422  (min 0.4968, max 0.539)


Column order alone swings PR-AUC by 0.04 on an identical feature set — as large as some of the effects this notebook is trying to measure. Every comparison from here on is therefore an average over several random column orderings of the same feature set, not a single-order point estimate.

In [9]:
def evaluate_stage_order_robust(cols, n_repeats=8, seed=0):
    """Repeats evaluate_ablation_stage() across n_repeats random orderings of the same column
    SET, to separate a real feature-set effect from the column-order sensitivity this pipeline
    turned out to have (see markdown above) - same set of columns each time, just shuffled.
    Returns mean/std/min/max PR-AUC across repeats, plus mean ROC-AUC."""
    rng = random.Random(seed)
    pr_aucs, roc_aucs = [], []
    for _ in range(n_repeats):
        shuffled = list(cols)
        rng.shuffle(shuffled)
        res = evaluate_ablation_stage(shuffled)
        pr_aucs.append(res['PR-AUC'])
        roc_aucs.append(res['ROC-AUC'])
    return {
        'PR-AUC mean': round(np.mean(pr_aucs), 4),
        'PR-AUC std':  round(np.std(pr_aucs), 4),
        'PR-AUC min':  round(min(pr_aucs), 4),
        'PR-AUC max':  round(max(pr_aucs), 4),
        'ROC-AUC mean': round(np.mean(roc_aucs), 4),
    }

### Strategies vs. the Current Model

The three points that answer this notebook's opening question, each an order-robust mean over 8 random column orderings: the current model, the current model with strategy scores added, and strategy scores alone.

In [10]:
headline_results = {
    'current model (no strategies)':   evaluate_stage_order_robust(list(X.columns)),
    'current model + strategy scores': evaluate_stage_order_robust(list(X_all.columns)),
    'strategy-only':                   evaluate_stage_order_robust(STRATEGY_COLS),
}

headline_df = pd.DataFrame(headline_results).T
baseline_mean = headline_df.loc['current model (no strategies)', 'PR-AUC mean']
baseline_std  = headline_df.loc['current model (no strategies)', 'PR-AUC std']
headline_df['Delta PR-AUC mean vs current model'] = (headline_df['PR-AUC mean'] - baseline_mean).round(4)
combined_std = headline_df['PR-AUC std'] + baseline_std
headline_df['Robust to order?'] = (headline_df['Delta PR-AUC mean vs current model'].abs() > combined_std).astype(object)
headline_df.loc['current model (no strategies)', 'Robust to order?'] = None
headline_df

,PR-AUC mean,PR-AUC std,PR-AUC min,PR-AUC max,ROC-AUC mean,Delta PR-AUC mean vs current model,Robust to order?
current model (no strategies),0.4467,0.0196,0.4166,0.4689,0.7024,0.0000,None
current model + strategy scores,0.5202,0.0177,0.4845,0.5433,0.7713,0.0735,True
strategy-only,0.3538,0.0079,0.3433,0.3645,0.6745,-0.0929,True


**Strategy scores add a real, order-robust +0.0735 PR-AUC on top of the current model — the clearest positive result in this notebook.** The current model alone reaches PR-AUC 0.4467 (± 0.0196 across orderings); adding the 8 strategy scores lifts it to 0.5202 (± 0.0177) — a delta that clears the combined order-noise band, so this isn't an ordering artifact. `strategy-only` (0.3538) sits between the no-skill baseline (0.233) and the current model — real signal on its own, but not a replacement for the existing feature set: strategies add more value as a supplement to the current model than they carry standalone.

The family-by-family breakdown further down reaches a similar full-feature-set estimate (0.5117) via a different construction path — building up from `strategy-only` family by family, rather than reordering the complete `X_all` set directly. The ~0.01 gap between the two is itself within the order-noise band established above; both point to the same conclusion.

### Strategy Scores as the Base: Adding Engineered Families One at a Time

The headline comparison above treats strategy scores as the *last* addition to an already-built engineered pipeline. This section flips that around, mirroring `04.2_baseline_comparison.ipynb`'s §8.8 feature-family methodology but with strategy scores as the fixed base instead of TF-IDF: starting from `strategy-only` (established above), how much does each *individual* engineered-feature family add on top, one at a time (**solo**), and how do the families compound when stacked cumulatively, added best-to-worst by their solo lift (**cumulative**)? Same fixed XGB+SMOTE architecture as the rest of this notebook.

Given this pipeline's column-order sensitivity (demonstrated earlier in this section), every PR-AUC in this section is an average over 8 random internal column orderings of the same feature set, not a single-order point estimate — so a family's reported lift reflects the feature set itself, not an arbitrary ordering artifact.

Families follow `04.2`'s own 8-family partition of the engineered features, plus a 9th — `length` (`word_count` alone). `04.2` excluded `word_count` from its family partition entirely (it wasn't part of that notebook's engineered feature set); `04.3`'s closing section formalized it as a feature, so it's part of `X` here (see the Setup cell above) and needs a family of its own. It gets a dedicated one rather than being absorbed into an existing family because `04.2`/`04.3` established it as the single strongest engineered signal in the whole project — worth tracking on its own rather than letting it disappear anonymously inside a larger bucket.

In [11]:
comment_tone_cols = [c for c in tone_df.columns if not c.startswith('tone_google_op_')]
op_tone_cols = [c for c in tone_df.columns if c.startswith('tone_google_op_')]

OLD_FEATURE_FAMILIES = {
    'length':             ['word_count'],
    'sentiment':          ['sentiment_llm'],
    'tone/style':         comment_tone_cols + ['is_formal', 'adj_adv_ratio'],
    'readability':        ['fk_grade_level', 'fk_reading_ease'],
    'evidence use':       ['evidence_count', 'use_of_persuasive_lang'],
    'ethos/pathos/logos': ['ethos_llm', 'pathos_llm', 'logos_llm'],
    'OP features':        op_tone_cols + ['is_formal_op', 'adj_adv_ratio_op', 'fk_grade_level_op',
                                           'evidence_count_op', 'sentiment_llm_op', 'ethos_llm_op',
                                           'pathos_llm_op', 'logos_llm_op'],
    'diff/match':         ['sentiment_diff', 'ethos_diff', 'pathos_diff', 'logos_diff',
                            'adj_adv_ratio_diff', 'fk_grade_level_diff', 'evidence_count_diff'] + match_features,
    'embeddings':         embedding_features,
}

# Must exactly partition X's columns - no column left out, none double-counted.
all_family_cols = [c for cols in OLD_FEATURE_FAMILIES.values() for c in cols]
assert len(all_family_cols) == len(set(all_family_cols)), 'a column appears in more than one family'
assert set(all_family_cols) == set(X.columns), f'families do not partition X exactly: missing {set(X.columns) - set(all_family_cols)}, extra {set(all_family_cols) - set(X.columns)}'

for name, cols in OLD_FEATURE_FAMILIES.items():
    print(f'{name:20s}: {len(cols):3d} columns')

length              :   1 columns
sentiment           :   1 columns
tone/style          :   7 columns
readability         :   2 columns
evidence use        :   2 columns
ethos/pathos/logos  :   3 columns
OP features         :  13 columns
diff/match          :  13 columns
embeddings          : 187 columns


In [12]:
strategy_alone_robust = evaluate_stage_order_robust(STRATEGY_COLS)
print(f"strategy-only (base), 8 random orderings: "
      f"PR-AUC {strategy_alone_robust['PR-AUC mean']} ± {strategy_alone_robust['PR-AUC std']} "
      f"[{strategy_alone_robust['PR-AUC min']}, {strategy_alone_robust['PR-AUC max']}]")

solo_family_results = {'strategy-only': strategy_alone_robust}
for family, cols in OLD_FEATURE_FAMILIES.items():
    solo_family_results[f'+ {family}'] = evaluate_stage_order_robust(STRATEGY_COLS + cols)

solo_family_df = pd.DataFrame(solo_family_results).T
solo_family_df['Delta PR-AUC vs strategy-only'] = (solo_family_df['PR-AUC mean'] - strategy_alone_robust['PR-AUC mean']).round(4)
# 'Robust' = the delta exceeds the combined one-sigma noise band of the two means being compared -
# a lightweight check, not a formal significance test, but enough to flag deltas that are order-noise-sized.
combined_std = solo_family_df['PR-AUC std'] + strategy_alone_robust['PR-AUC std']
solo_family_df['Robust to order?'] = (solo_family_df['Delta PR-AUC vs strategy-only'].abs() > combined_std).astype(object)
solo_family_df.loc['strategy-only', 'Robust to order?'] = None
solo_family_df = solo_family_df.sort_values('PR-AUC mean', ascending=False)
solo_family_df

strategy-only (base), 8 random orderings: PR-AUC 0.3538 ± 0.0079 [0.3433, 0.3645]


,PR-AUC mean,PR-AUC std,PR-AUC min,PR-AUC max,ROC-AUC mean,Delta PR-AUC vs strategy-only,Robust to order?
+ length,0.5379,0.0109,0.5199,0.5581,0.7733,0.1841,True
+ embeddings,0.3902,0.0142,0.3676,0.4087,0.6940,0.0364,True
+ sentiment,0.3869,0.0079,0.3695,0.3944,0.7098,0.0331,True
+ tone/style,0.3776,0.0111,0.3600,0.3936,0.6824,0.0238,True
+ diff/match,0.3745,0.0133,0.3538,0.3893,0.7044,0.0207,False
+ OP features,0.3703,0.0135,0.3504,0.3922,0.6688,0.0165,False
+ readability,0.3643,0.0054,0.3559,0.3754,0.6733,0.0105,False
strategy-only,0.3538,0.0079,0.3433,0.3645,0.6745,0.0000,None
+ evidence use,0.3533,0.0110,0.3421,0.3719,0.6637,-0.0005,False
+ ethos/pathos/logos,0.3298,0.0066,0.3158,0.3361,0.6407,-0.0240,True


In [13]:
# Cumulative build order = solo-lift ranking (best first, by mean PR-AUC), read directly off
# solo_family_df above - same convention as 04.2 §8.8. Each step's SET of columns is fixed by
# this order (which family joins when), but its PR-AUC is itself an order-robust mean across
# 8 random internal orderings of that set - not a single-order point estimate.
family_order_by_solo = [name[2:] for name in solo_family_df.index if name.startswith('+ ')]
print('Cumulative build order (best-to-worst by mean solo PR-AUC):', family_order_by_solo)

cumulative_family_results = {'strategy-only': strategy_alone_robust}
cumulative_cols = list(STRATEGY_COLS)
for family in family_order_by_solo:
    cumulative_cols = cumulative_cols + OLD_FEATURE_FAMILIES[family]
    cumulative_family_results[f'+ {family}'] = evaluate_stage_order_robust(cumulative_cols)

cumulative_family_df = pd.DataFrame(cumulative_family_results).T
cumulative_family_df['Delta PR-AUC mean vs previous row'] = cumulative_family_df['PR-AUC mean'].diff().round(4)
cumulative_family_df

Cumulative build order (best-to-worst by mean solo PR-AUC): ['length', 'embeddings', 'sentiment', 'tone/style', 'diff/match', 'OP features', 'readability', 'evidence use', 'ethos/pathos/logos']


,PR-AUC mean,PR-AUC std,PR-AUC min,PR-AUC max,ROC-AUC mean,Delta PR-AUC mean vs previous row
strategy-only,0.3538,0.0079,0.3433,0.3645,0.6745,NaN
+ length,0.5379,0.0109,0.5199,0.5581,0.7733,0.1841
+ embeddings,0.4954,0.0092,0.4877,0.5078,0.7566,-0.0425
+ sentiment,0.4999,0.0175,0.4785,0.5262,0.7610,0.0045
+ tone/style,0.5122,0.0104,0.4995,0.5263,0.7627,0.0123
+ diff/match,0.5197,0.0087,0.5104,0.5290,0.7614,0.0075
+ OP features,0.5031,0.0207,0.4680,0.5250,0.7630,-0.0166
+ readability,0.5099,0.0123,0.4908,0.5291,0.7626,0.0068
+ evidence use,0.4946,0.0160,0.4723,0.5226,0.7561,-0.0153
+ ethos/pathos/logos,0.5117,0.0220,0.4711,0.5449,0.7610,0.0171


**Averaging over column order changes the picture — this was worth doing.** Only 5 of the 9 families show a solo effect that clears the order-noise band: `length` (+0.1841, no surprise), `embeddings` (+0.0364), `sentiment` (+0.0331), and `tone/style` (+0.0238) are robustly positive; `ethos/pathos/logos` (-0.0240) is robustly negative. The other four — `diff/match`, `OP features`, `readability`, `evidence use` — all have deltas smaller than their own order-induced spread, so nothing can be said about them either way from this check.

`embeddings` is the clearest case where averaging mattered: the single-order pass (before this robustness check existed) had it looking like one of the weaker additions; averaged over 8 random orderings it's actually the second-strongest robust family. `ethos/pathos/logos` stays negative either way, so that finding — redundant with strategy scores rather than order noise — holds up. Consistent with the earlier point-estimate read, the *cumulative* build still isn't monotonic: `+ embeddings` costs -0.0425 PR-AUC in the cumulative sequence despite being robustly positive solo — a real interaction/redundancy effect with `length` and `sentiment` (already added by that point), not measurement noise, since this is order-averaged too. The final row — strategy scores plus every engineered family — reaches PR-AUC 0.5117, now a stable, order-averaged estimate rather than a single arbitrary-order point estimate.

## Re-Selecting the Model on the Best Feature Combination

The family-by-family comparison above identified `strategy scores + word_count` as the strongest feature combination found in this notebook — order-robust mean PR-AUC 0.5379, ahead of every larger combination tried, including the full engineered feature set. Rather than re-testing multiple legacy feature-set variants, this section fixes the feature set to that best combination and asks a narrower question: given these features, is `XGB (over)` still the optimal architecture, or does a different classifier or resampling strategy win once the feature set itself is no longer a variable?

In [14]:
BEST_FEATURE_COLS = STRATEGY_COLS + OLD_FEATURE_FAMILIES['length']
print(f'Best feature combination: {len(BEST_FEATURE_COLS)} columns - {BEST_FEATURE_COLS}')

Best feature combination: 9 columns - ['justification', 'concession', 'evidence', 'refutation', 'personal_narrative', 'call_to_action', 'simplification', 'question_asking', 'word_count']


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler

def make_pipeline_final(clf, resampling='none'):
    steps = [('scaler', RobustScaler())]
    if resampling == 'under':
        steps.append(('sampler', RandomUnderSampler(random_state=42)))
    elif resampling == 'over':
        steps.append(('sampler', SMOTE(random_state=42)))
    steps.append(('clf', clf))
    return Pipeline(steps)

pipelines_final = {
    'LogReg (none)':  make_pipeline_final(LogisticRegression(random_state=42, max_iter=1000), 'none'),
    'LogReg (under)': make_pipeline_final(LogisticRegression(random_state=42, max_iter=1000), 'under'),
    'LogReg (over)':  make_pipeline_final(LogisticRegression(random_state=42, max_iter=1000), 'over'),
    'RF (none)':      make_pipeline_final(RandomForestClassifier(random_state=42, n_estimators=100), 'none'),
    'RF (under)':     make_pipeline_final(RandomForestClassifier(random_state=42, n_estimators=100), 'under'),
    'RF (over)':      make_pipeline_final(RandomForestClassifier(random_state=42, n_estimators=100), 'over'),
    'XGB (none)':     make_pipeline_final(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'none'),
    'XGB (under)':    make_pipeline_final(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'under'),
    'XGB (over)':     make_pipeline_final(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'over'),
}
print(f'{len(pipelines_final)} pipelines defined')

9 pipelines defined


In [16]:
# Thread-disjoint inner-train/validation split, carved from train_idx - same convention used
# throughout this notebook, so model selection never touches the held-out test set.
train_groups_final = groups.iloc[train_idx].reset_index(drop=True)
y_train_final_r    = y_train.reset_index(drop=True)

gss_val_final = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
inner_pos, val_pos = next(gss_val_final.split(np.zeros((len(train_idx), 1)), y_train_final_r, groups=train_groups_final))

inner_idx_final = np.array(train_idx)[inner_pos]
val_idx_final   = np.array(train_idx)[val_pos]

y_inner_final = y.loc[inner_idx_final]
y_val_final   = y.loc[val_idx_final]

inner_threads_final = set(groups.loc[inner_idx_final])
val_threads_final   = set(groups.loc[val_idx_final])
assert len(inner_threads_final & val_threads_final) == 0
print(f'Inner train     : {len(inner_idx_final)} comments from {len(inner_threads_final)} threads')
print(f'Validation      : {len(val_idx_final)} comments from {len(val_threads_final)} threads')
print(f'Test (held out) : {len(test_idx)} comments from {groups.loc[test_idx].nunique()} threads')

Inner train     : 1671 comments from 256 threads
Validation      : 490 comments from 64 threads
Test (held out) : 592 comments from 80 threads


In [17]:
X_best_inner = X_all[BEST_FEATURE_COLS].loc[inner_idx_final]
X_best_val   = X_all[BEST_FEATURE_COLS].loc[val_idx_final]

val_metrics_final = {}
for model_name, pipe in pipelines_final.items():
    pipe.fit(X_best_inner, y_inner_final)
    proba_val = pipe.predict_proba(X_best_val)[:, 1]
    val_metrics_final[model_name] = {
        'PR-AUC':  round(average_precision_score(y_val_final, proba_val), 4),
        'ROC-AUC': round(roc_auc_score(y_val_final, proba_val), 4),
    }

val_df_final = pd.DataFrame(val_metrics_final).T.sort_values('PR-AUC', ascending=False)
print(f'Validation results across {len(val_metrics_final)} (classifier x resampling) combinations, '
      f'feature set fixed to the best combination found above:')
val_df_final

Validation results across 9 (classifier x resampling) combinations, feature set fixed to the best combination found above:


,PR-AUC,ROC-AUC
RF (over),0.4331,0.7319
RF (none),0.4325,0.7422
XGB (over),0.4133,0.7015
RF (under),0.3914,0.7387
XGB (none),0.3777,0.6897
LogReg (none),0.3452,0.6987
XGB (under),0.3439,0.7003
LogReg (under),0.3433,0.6976
LogReg (over),0.3378,0.6832


In [18]:
best_model_name_final = val_df_final.index[0]
best_pipe_final = pipelines_final[best_model_name_final]
X_best_train = X_all[BEST_FEATURE_COLS].loc[train_idx]
X_best_test  = X_all[BEST_FEATURE_COLS].loc[test_idx]

best_pipe_final.fit(X_best_train, y_train)
proba_test_final = best_pipe_final.predict_proba(X_best_test)[:, 1]
pred_test_final  = best_pipe_final.predict(X_best_test)

pr_auc_test_final  = average_precision_score(y_test, proba_test_final)
roc_auc_test_final = roc_auc_score(y_test, proba_test_final)
f1_test_final       = f1_score(y_test, pred_test_final, zero_division=0)

# Reference: the same best feature combination, but with the architecture this notebook has used
# throughout (XGB + SMOTE) rather than whichever architecture won the grid above - isolates the
# effect of re-selecting the architecture, holding the feature set fixed.
reference_pipe = make_selected_pipe()
reference_pipe.fit(X_best_train, y_train)
proba_reference = reference_pipe.predict_proba(X_best_test)[:, 1]
pred_reference  = reference_pipe.predict(X_best_test)

pr_auc_reference  = average_precision_score(y_test, proba_reference)
roc_auc_reference = roc_auc_score(y_test, proba_reference)
f1_reference       = f1_score(y_test, pred_reference, zero_division=0)

no_skill_baseline = y_test.mean()
print(f'No-skill PR-AUC baseline (test-fold positive rate): {no_skill_baseline:.4f}')

selection_comparison = pd.DataFrame({
    'Test PR-AUC':  [pr_auc_reference, pr_auc_test_final],
    'Test ROC-AUC': [roc_auc_reference, roc_auc_test_final],
    'Test F1 @ 0.5':[f1_reference, f1_test_final],
}, index=['XGB (over) - this notebook\'s default architecture', best_model_name_final + ' - grid winner'])
selection_comparison.round(4)

No-skill PR-AUC baseline (test-fold positive rate): 0.2382


,Test PR-AUC,Test ROC-AUC,Test F1 @ 0.5
XGB (over) - this notebook's default architecture,0.5399,0.7789,0.4854
RF (over) - grid winner,0.5387,0.7832,0.4659


**Random Forest edges out XGBoost on validation, but the two are indistinguishable on the held-out test set.** `RF (over)` tops the validation grid (PR-AUC 0.4331), narrowly ahead of `RF (none)` (0.4325) and more clearly ahead of `XGB (over)` (0.4133) — with only 9 columns, resampling and even the choice between Random Forest and XGBoost matter far less than they did on the full 237-column feature set. On the test set, however, this notebook's default architecture (`XGB (over)`) actually scores marginally *higher* (0.5399) than the nominal grid winner (0.5387) — a reversal, though well within noise given how close the two are. There is no compelling case here to move away from `XGB (over)`.

The more notable result is the absolute test PR-AUC itself: 0.5399. Unlike ROC-AUC, PR-AUC's no-skill baseline is not 0.5 but the positive class rate — 0.2382 on this test fold (printed above) — so 0.5399 represents roughly a **2.3x lift over random**, not a marginal one. It is also the best PR-AUC reached anywhere in this notebook on the full scored population (the length-matched benchmark below uses a smaller, rebalanced sample and is not directly comparable), using only 9 columns — 8 strategy scores plus `word_count` — well ahead of every larger combination tried above, including the full engineered feature set. Concentrating on the feature combination that the ablation identified as strongest mattered far more than which classifier or resampling strategy was paired with it.

## Length-Controlled Benchmark

The last question is whether strategy scores explain signal that *survives* length control, not just whether they help on the regular ≥30-word dataset above — `04.2`'s length-signal discovery and `04.3`'s length-matched resampling (§8.11) both show that a large share of what looks like engineered-feature or TF-IDF signal is really comment length in disguise.

`04.3`'s strict 1:1 length-matched sample can't be reused directly here: it was built from the *full* 3,971-row dataset, decile-matching non-convincing comments' length to convincing comments' length — many of the matched non-convincing rows fall well under the 30-word floor, where no strategy score exists (`05` only scored the ≥30-word-or-convincing population). So the same matching procedure is re-run here, restricted to this notebook's own 2,753-row scored population — the resulting sample is smaller, but every row in it has a strategy score.

In [19]:
from scipy.stats import pointbiserialr

r_full, p_full = pointbiserialr(comments_df['is_convincing'], comments_df['word_count'])
print(f'word_count correlation, scored population (n={len(comments_df)}): r={r_full:.4f}, p={p_full:.2e}')

n_bins = 10
conv_df    = comments_df[comments_df['is_convincing'] == 1].copy()
nonconv_df = comments_df[comments_df['is_convincing'] == 0].copy()

bin_edges = pd.qcut(conv_df['word_count'], n_bins, duplicates='drop', retbins=True)[1]
conv_df['length_bin']    = pd.cut(conv_df['word_count'], bins=bin_edges, include_lowest=True)
nonconv_df['length_bin'] = pd.cut(nonconv_df['word_count'], bins=bin_edges, include_lowest=True)

matched_parts = []
for bin_label, group in conv_df.groupby('length_bin', observed=True):
    pool = nonconv_df[nonconv_df['length_bin'] == bin_label]
    take_n = min(len(group), len(pool))
    if take_n > 0:
        matched_parts.append(pool.sample(n=take_n, random_state=42))

matched_nonconv_df = pd.concat(matched_parts) if matched_parts else nonconv_df.iloc[:0]
comments_df_matched = pd.concat([conv_df, matched_nonconv_df]).drop(columns='length_bin').sort_index()

r_matched, p_matched = pointbiserialr(comments_df_matched['is_convincing'], comments_df_matched['word_count'])
print(f'\nStrict 1:1 length-matched sample (within the scored population):')
print(f'  {len(comments_df_matched)} comments ({int(comments_df_matched["is_convincing"].sum())} convincing, '
      f'positive rate {comments_df_matched["is_convincing"].mean():.3f})')
print(f'  word_count correlation after matching: r={r_matched:.4f}, p={p_matched:.2e}  (unmatched: r={r_full:.4f})')

word_count correlation, scored population (n=2753): r=0.2882, p=8.70e-54

Strict 1:1 length-matched sample (within the scored population):
  1235 comments (641 convincing, positive rate 0.519)
  word_count correlation after matching: r=0.0608, p=3.27e-02  (unmatched: r=0.2882)


This comparison isolates *only* the strategy-score effect: both arms use `X.columns` ('trusted', no untrusted features) rather than the 9-column `strategy scores + word_count` combination the "Re-Selecting the Model" section above evaluated. As a result, the "without strategy" PR-AUC below (0.4059) isn't comparable to that section's 0.5399 test PR-AUC — both are correct, they're just measured on different feature sets answering different questions.

In [20]:
def evaluate_on_sample(sample_df, cols, seed=42):
    """Thread-grouped split within sample_df's rows, fixed selected-pipeline architecture,
    given feature columns only. Mirrors 04.3's evaluate_length_sample, generalized to any
    column subset so it can compare with/without strategy scores on the same sample."""
    idx = sample_df.index
    y_s = comments_df.loc[idx, 'is_convincing']
    groups_s = comments_df.loc[idx, 'thread_id']

    gss_s = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_pos, te_pos = next(gss_s.split(np.zeros((len(idx), 1)), y_s, groups=groups_s))
    tr_idx, te_idx = idx[tr_pos], idx[te_pos]
    y_tr, y_te = y_s.loc[tr_idx], y_s.loc[te_idx]

    pipe = clone(make_selected_pipe())
    pipe.fit(X_all.loc[tr_idx, cols], y_tr)
    proba = pipe.predict_proba(X_all.loc[te_idx, cols])[:, 1]

    return {
        'n_total': len(idx), 'n_train': len(tr_idx), 'n_test': len(te_idx),
        'test_positive_rate': float(y_te.mean()),
        'PR-AUC': average_precision_score(y_te, proba),
        'ROC-AUC': roc_auc_score(y_te, proba),
    }

# Same two column sets compared in the model-selection section above: the champion architecture's
# feature set with and without the 8 strategy scores (both 'trusted', no untrusted features -
# the winning feature set once strategy scores are available, per the previous section).
cols_without_strategy = list(X.columns)
cols_with_strategy    = list(X.columns) + STRATEGY_COLS

results_matched_without = evaluate_on_sample(comments_df_matched, cols_without_strategy)
results_matched_with    = evaluate_on_sample(comments_df_matched, cols_with_strategy)
results_full_without    = evaluate_on_sample(comments_df, cols_without_strategy)
results_full_with       = evaluate_on_sample(comments_df, cols_with_strategy)

In [21]:
# Lift over no-skill = PR-AUC minus THIS ROW'S OWN test-fold positive rate - the correct,
# sample-independent comparison (04.3 §8.11's correction: the whole-sample rate drifts from the
# actual test-fold rate on a split this small).
length_benchmark_rows = []
for sample_name, res_without, res_with in [
    ('Full scored population (n=2753)', results_full_without, results_full_with),
    ('Length-matched (within scored population)', results_matched_without, results_matched_with),
]:
    length_benchmark_rows.append({
        'sample': sample_name,
        'n': res_without['n_total'],
        'test positive rate': res_without['test_positive_rate'],
        'PR-AUC without strategy': res_without['PR-AUC'],
        'PR-AUC with strategy':    res_with['PR-AUC'],
        'lift without strategy':   res_without['PR-AUC'] - res_without['test_positive_rate'],
        'lift with strategy':      res_with['PR-AUC'] - res_with['test_positive_rate'],
    })

length_benchmark_df = pd.DataFrame(length_benchmark_rows).set_index('sample').round(4)
length_benchmark_df['strategy-score gain (PR-AUC)'] = (length_benchmark_df['PR-AUC with strategy'] - length_benchmark_df['PR-AUC without strategy']).round(4)
length_benchmark_df

,n,test positive rate,PR-AUC without strategy,PR-AUC with strategy,lift without strategy,lift with strategy,strategy-score gain (PR-AUC)
sample,,,,,,,
Full scored population (n=2753),2753,0.2382,0.4059,0.4968,0.1678,0.2587,0.0909
Length-matched (within scored population),1235,0.5132,0.5559,0.6384,0.0427,0.1252,0.0825


**Strategy scores survive length control almost undiminished — unlike everything else measured so far in this project.** Both arms collapse sharply once length is matched, as expected from `04.3` §8.11: lift-over-no-skill without strategy scores drops from 0.1678 to 0.0427 (a 75% reduction), and with strategy scores from 0.2587 to 0.1252 (a 52% reduction) — length is still doing most of the work either way.

But the *strategy-score gain itself* — the difference the 8 scores add on top of the engineered features, holding everything else fixed — barely moves: **0.0909 PR-AUC on the full population vs. 0.0825 on the length-matched sample, a 91% retention.** For comparison, `04.3` §8.11 found `word_count`'s own lift collapsed to 18% of its full-dataset value under the same kind of strict matching, and the engineered pipeline's to 25%. Strategy scores behave nothing like that here — whatever they're capturing looks like it's substantially independent of comment length, not another length proxy riding along with everything else.

## Summary

Comment length remains the dominant signal in this dataset: `word_count` alone outperforms every other individual feature or family considered, by a substantial margin. Strategy scores add a real but modest improvement on top of the existing engineered pipeline — on the order of a few points of PR-AUC, depending on the exact comparison — rather than a large gain. At the level of individual feature families, only a minority of additions (length, embeddings, sentiment/tone) produce a lift distinguishable from the column-order noise characterized above; most others cannot be confidently said to help or hurt.

The result of greatest substantive interest, however, is the length-controlled benchmark above: once comment length is controlled for, the predictive contribution of most other features collapses sharply — lift-over-no-skill without strategy scores drops from 0.1678 to 0.0427 (a 75% reduction) — but the strategy scores retain the large majority of theirs (0.0909 to 0.0825 PR-AUC, a 91% retention). For comparison, `04.3` found `word_count`'s own lift collapsed to just 18% of its full-dataset value under the same kind of matching. This suggests the strategy scores are not primarily a proxy for comment length, unlike much of the signal examined elsewhere in this project — they appear to capture something about *how* an argument is constructed, rather than simply *how much* was written. Explaining *which* strategies drive this (SHAP beeswarm, per-strategy Mann-Whitney effect sizes, repeated on the length-matched sample) is the natural next step.